# Movement markers from SLEAP 

In [52]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.io import savemat
sys.path.append("/home/zms24/Desktop") 
import PyalData.pyaldata as pyal # type:ignore

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from tools.curbd import curbd
import tools.dsp.preprocessing as preprocessing

importlib.reload(preprocessing)

preprocess = preprocessing.preprocess  # (Re)assign the function if needed
np.random.seed(62)

### Import data

In [53]:
data_dir = "/data/raw/M062/M062_2025_03_21_14_00"
mat_file_0= "M062_2025_03_21_14_00_pyaldata_0.mat"
mat_file_1= "M062_2025_03_21_14_00_pyaldata_1.mat"
mat_file_2= "M062_2025_03_21_14_00_pyaldata_2.mat"

fname0 = os.path.join(data_dir, mat_file_0)
fname1 = os.path.join(data_dir, mat_file_1)
fname2 = os.path.join(data_dir, mat_file_2)

df0 = pyal.mat2dataframe(fname0, shift_idx_fields=True)
df1 = pyal.mat2dataframe(fname1, shift_idx_fields=True)
df2 = pyal.mat2dataframe(fname2, shift_idx_fields=True)
df = pd.concat([df0, df1, df2], ignore_index=True)
df = df.drop(columns="all_spikes") # the content is incorrect

field values_before_camera_trigger could not be converted to int.
field idx_before_camera_trigger could not be converted to int.
array field VAL_KSLabel could not be converted to int.
array field SSp_KSLabel could not be converted to int.
array field all_KSLabel could not be converted to int.
array field CP_KSLabel could not be converted to int.
array field MOp_KSLabel could not be converted to int.
field values_Sol_direction could not be converted to int.
field idx_Sol_direction could not be converted to int.
field values_Sol_duration could not be converted to int.
field idx_Sol_duration could not be converted to int.
field idx_sol_on could not be converted to int.
array field VAL_KSLabel could not be converted to int.
array field SSp_KSLabel could not be converted to int.
array field all_KSLabel could not be converted to int.
array field CP_KSLabel could not be converted to int.
array field MOp_KSLabel could not be converted to int.
field values_Sol_direction could not be converted t

### Preprocess

In [54]:
df_ = preprocess(df, only_trials=False, repair_time_varying_fields=['MotSen1_X', 'MotSen1_Y'])
BIN_SIZE = df_['bin_size'][0]
# get 'all_rates' column
areas =[ "MOp_rates", "SSp_rates", "CP_rates", "VAL_rates"]
df_ = pyal.merge_signals(df_, areas, "all_rates")
areas.append("all_rates")

Repairing columns ['MotSen1_X', 'MotSen1_Y']
Extending index to 47999 in trial: free and id: 0, inserting NaN.
Extending index to 99 in trial: intertrial and id: 1, inserting NaN.
Extending index to 599 in trial: trial and id: 2, inserting NaN.
Extending index to 299 in trial: intertrial and id: 3, inserting NaN.
Extending index to 599 in trial: trial and id: 4, inserting NaN.
Extending index to 99 in trial: intertrial and id: 5, inserting NaN.
Extending index to 599 in trial: trial and id: 6, inserting NaN.
Extending index to 299 in trial: intertrial and id: 7, inserting NaN.
Extending index to 599 in trial: trial and id: 8, inserting NaN.
Extending index to 499 in trial: intertrial and id: 9, inserting NaN.
Extending index to 599 in trial: trial and id: 10, inserting NaN.
Extending index to 499 in trial: intertrial and id: 11, inserting NaN.
Extending index to 599 in trial: trial and id: 12, inserting NaN.
Extending index to 299 in trial: intertrial and id: 13, inserting NaN.
Extendi

/home/zms24/Desktop/PyalData/pyaldata/firing_rates.py:108: UserWarning: Assuming spikes are actually spikes and dividing by bin size.
  utils.warnings.warn(
/home/zms24/Desktop/PyalData/pyaldata/firing_rates.py:108: UserWarning: Assuming spikes are actually spikes and dividing by bin size.
  utils.warnings.warn(
/home/zms24/Desktop/PyalData/pyaldata/firing_rates.py:108: UserWarning: Assuming spikes are actually spikes and dividing by bin size.
  utils.warnings.warn(
/home/zms24/Desktop/PyalData/pyaldata/firing_rates.py:108: UserWarning: Assuming spikes are actually spikes and dividing by bin size.
  utils.warnings.warn(


Combined every 3 bins
Resulting VAL_spikes ephys data shape is (NxT): (122, 16000)
Resulting SSp_spikes ephys data shape is (NxT): (66, 16000)
Resulting CP_spikes ephys data shape is (NxT): (280, 16000)
Resulting MOp_spikes ephys data shape is (NxT): (179, 16000)


In [58]:
# correct trial length - this is an error in pyaldata
df_['trial_length'] = (df_['trial_length'] / (BIN_SIZE * 100)).astype(int)

# === Metadata ===
session_id = mat_file_0.replace("_pyaldata_0.mat", "")
mouse = session_id.split('_')[0]
perturb_time_idx = df_.idx_sol_on[2]
perturb_time_sec = perturb_time_idx * BIN_SIZE

sol_angles = sorted(df_[df_['trial_name']=='trial'].values_Sol_direction.unique())
trial_labels = [f"solenoid {angle}" for angle in sol_angles]
num_trials = len(df_)

print(f"Mouse: {mouse}")
print(f"Number of trials: {num_trials}")
print(f"Perturbation time (bins): {perturb_time_idx}, ({perturb_time_sec:.2f} sec)")

Mouse: M062
Number of trials: 708
Perturbation time (bins): 66, (1.98 sec)


## Determine gait cycles

In [ ]:
def detect_gait_cycles(angle_data, distance=4, peak_height=80, peak_threshold=0.3):
    """
    Detects gait cycles in a 1D joint angle signal based on peak detection.

    Parameters:
    - angle_data : list or np.ndarray
        1D sequence of joint angles.
    - distance : int
        Minimum distance between peaks (in frames).
    - peak_height : float
        Minimum height of detected peaks.
    - peak_threshold : float
        Threshold to use for peak detection.

    Returns:
    - gait_cycles : list of tuples
        List of (start_idx, end_idx) tuples representing gait cycles.
    """
    angle_seq = np.array(angle_data)

    # Detect peaks
    peaks, _ = find_peaks(angle_seq, distance=distance, height=peak_height, threshold=peak_threshold)

    # Define gait cycles between successive peaks
    gait_cycles = [(peaks[i], peaks[i + 1]) for i in range(len(peaks) - 1)]

    return gait_cycles

In [ ]:
# Initialize empty list to collect rows
gait_rows = []
gait_id_counter = 0

for trial_idx, trial_row in df_.iterrows():
    angle_data = trial_row['right_ankle_angle']
    
    # Detect gait cycles
    gait_cycles = detect_gait_cycles(angle_data, distance=4, )
    
    sol_on_idx = trial_row['idx_sol_on']

    # Handle empty arrays or single-element arrays
    if isinstance(sol_on_idx, (list, np.ndarray)):
        if len(sol_on_idx) == 0 or np.array(sol_on_idx).size == 0:
            sol_on_idx = None
        elif np.array(sol_on_idx).size == 1:
            sol_on_idx = np.array(sol_on_idx).item()

    for start, end in gait_cycles:
        # Only skip cycles after perturbation if sol_on_idx is defined
        if sol_on_idx is not None and end >= sol_on_idx:
            continue

        # Get neural data slices during the gait cycle
        gait_VAL = trial_row['VAL_rates'][start:end]
        gait_SSp = trial_row['SSp_rates'][start:end]
        gait_CP = trial_row['CP_rates'][start:end]
        gait_MOp = trial_row['MOp_rates'][start:end]
        gait_len = end - start
        
        # Append new row
        gait_rows.append({
            'gait_id': gait_id_counter,
            'trial_id': trial_row['trial_id'],
            'trial_name': trial_row['trial_name'],
            'trial_length': trial_row['trial_length'],
            'gait_length': gait_len,
            'bin_size': trial_row['bin_size'],
            'values_Sol_direction': trial_row['values_Sol_direction'],
            'sol_level_id': trial_row['sol_level_id'],
            'VAL_rates': gait_VAL,
            'SSp_rates': gait_SSp,
            'CP_rates': gait_CP,
            'MOp_rates': gait_MOp,
            'gait_start_idx': start,
            'gait_end_idx': end
        })
        
        gait_id_counter += 1

# Create final dataframe
df_gait = pd.DataFrame(gait_rows)

In [60]:
df_gait.head()

,gait_id,trial_id,trial_name,trial_length,gait_length,bin_size,values_Sol_direction,sol_level_id,VAL_rates,SSp_rates,CP_rates,MOp_rates,gait_start_idx,gait_end_idx
0,0,0,free,16000,4,0.03,[],NaN,"[[0.034451418, 0.0, 0.0, 0.0, 0.0, 0.0, 7.9468...","[[0.20841894, 25.337986, 8.507438, 0.0, 10.720...","[[0.0, 0.0, 0.0, 0.0, 0.00031967516, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 5.5645814, 0.20841894, 0...",1,8
1,1,0,free,16000,4,0.03,[],NaN,"[[5.3604555, 0.0, 0.034451418, 0.0, 0.0, 0.0, ...","[[3.4700277, 26.24538, 3.4703474, 0.0, 30.0095...","[[0.0, 0.0, 0.0, 0.0, 7.836184, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.0, 2.6248064, 2.594328, 0.0...",8,15
2,2,0,free,16000,4,0.03,[],NaN,"[[25.124203, 0.0, 5.321711, 0.0, 0.0, 0.0, 0.0...","[[1.7593459, 26.30891, 1.0880919, 0.0, 8.78918...","[[0.0, 0.0, 0.0, 0.0, 3.474001, 0.0, 0.0, 0.0,...","[[0.0, 0.0, 0.0, 0.034771092, 0.0, 7.627765, 0...",15,22
3,3,0,free,16000,4,0.03,[],NaN,"[[16.718544, 0.0, 0.0, 5.321711, 0.0, 0.0, 0.0...","[[0.87967294, 24.085127, 0.0, 0.0, 14.114089, ...","[[0.0, 0.0, 0.0, 2.590355, 0.87967294, 0.0, 0....","[[0.0, 0.0, 0.0, 13.158215, 2.6248064, 5.32203...",22,31
4,4,0,free,16000,4,0.03,[],NaN,"[[7.632058, 0.0, 0.00031967516, 0.00031967516,...","[[0.0, 22.218496, 7.633836, 1.0880919, 5.36442...","[[0.0, 0.0, 7.627765, 0.003973114, 0.0, 0.0, 0...","[[0.0, 0.0, 0.0, 5.53013, 0.883646, 0.00031967...",31,38


## For the RNN avergaing, I need to asign some fake solenoid angles

In [ ]:
## for the trials the solenoid angles are 0 - 11
## lets make the free period solenoid angles 12 - 15
## lets make the intertrial solenoid angles 16 - 19
## and lastly the 2nd free running period 20 - 23

for idx, row in df_gait.iterrows():
    if row['trial_name'] == 'free' and row['trial_id'] == 0: # first free period
        df_gait.at[idx, 'values_Sol_direction'] = np.random.randint(12, 16)
    elif row['trial_name'] == 'intertrial': # intertrial period
        df_gait.at[idx, 'values_Sol_direction'] = np.random.randint(16, 20)
    elif row['trial_name'] == 'free' and row['trial_id'] != 0: # last free period
        df_gait.at[idx, 'values_Sol_direction'] = np.random.randint(20, 24)

In [67]:
sol_angles = sorted(df_gait.values_Sol_direction.unique())

In [68]:
print(sol_angles)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


## We need to trim the trials to same length, otherwise we cannot avg them

In [ ]:
# trim all _rates columns
for idx in df_gait.index:
    for area in areas:
        arr = df_gait.at[idx, area]
        if arr is not None and isinstance(arr, np.ndarray):
            df_gait.at[idx, area] = arr[:4]
        else:
            print(f"Wrong spike array format at index: {idx}!")

# change gait_length column accordingly
for idx in df_gait.index:
    df_gait.at[idx, 'gait_length'] = len(df_gait.at[idx, 'MOp_rates'])